# Line Identification

[Chapter 5](05-arc-lamps.ipynb) introduced the arc frames; Chapter 6 straightened them. What
remains is to turn the emission lines they carry into (pixel, wavelength) pairs, the raw
material every wavelength solution is fitted to.

The work divides into two parts.

- **Detection**: locating the emission lines and measuring their centroids to sub-pixel
  precision. `specreduce.line_matching.find_arc_lines` does this. In the normal workflow it is
  not called directly: `WavelengthCalibration1D` runs it on the arc spectra it is given, through
  its `find_lines` method, as Chapter 10 shows. This chapter runs it by hand to look at what it
  returns. Nothing downstream is tied to it either way, because the calibration also accepts
  observed line positions measured with another detection routine, which serve it equally
  well.
- **Identification**: deciding which wavelength each detected line has. In the interactive
  workflow this is a human step. The detected lines, labeled with their pixel positions, are
  compared against reference figures of the same lamps through the same instrument setup, on
  which the lines are labeled with their wavelengths, and a handful of strong lines present in
  both are paired up.

This chapter does the detection, introduces the reference material, and walks through the
identification. It ends with a set of anchor lines that
[Chapter 10](10-wavelength-solution-interactive.ipynb) fits a solution to and then grows.
[Chapter 11](11-wavelength-solution-automated.ipynb) needs no anchors at all, because it starts
from prior knowledge of the solution rather than from the spectrum.

In [ ]:
import sys
import warnings

sys.path.append("../src")

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from specreduce.calibration_data import (
    get_available_line_catalogs,
    load_pypeit_calibration_lines,
)
from specreduce.extract import BoxcarExtract
from specreduce.line_matching import find_arc_lines
from specreduce.tilt_solution import TiltSolution
from specreduce.tracing import FlatTrace

from common import (
    ANCHOR_PIXELS,
    ANCHOR_WAVELENGTHS,
    GTC_HGAR,
    TILT_SOLUTION_FILE,
    fit_tilt_solution,
    read_data,
)

plt.rc("figure", figsize=(8, 2.4), dpi=110)
plt.rc("font", size=9)

arcs, lamps, obj = read_data()
nx = obj.data.shape[1]

## Where the previous chapters left off

Two lines of setup, so the chapter runs on its own: read Chapter 6's tilt solution and rectify the
arcs with it, then collapse each to 1D with the `BoxcarExtract` of Chapter 8. The three spectra
are what the rest of the chapter works with.

In [ ]:
reference = FlatTrace(arcs[0], 135)

if not TILT_SOLUTION_FILE.exists():  # normally written by Chapter 6
    fit_tilt_solution(arcs).to_asdf(TILT_SOLUTION_FILE)

solution = TiltSolution.from_asdf(TILT_SOLUTION_FILE)

rectified_arcs = [solution.resample(frame) for frame in arcs]


spectra = [
    BoxcarExtract(frame, reference, width=15).spectrum for frame in rectified_arcs
]

fig, axs = plt.subplots(3, figsize=(8, 5.0), sharex="all", constrained_layout=True)

for ax, lamp, spectrum in zip(axs, lamps, spectra):
    ax.plot(spectrum.flux.value, lw=0.6)
    ax.text(0.01, 0.9, lamp, transform=ax.transAxes, va="top", size=8)
    ax.set(yscale="log", ylim=(30, 1e7), ylabel="Flux [e$^-$]")

axs[-1].set(xlim=(0, nx - 1), xlabel="Dispersion [pix]");

The lines sit on a floor that is not zero. It is the lamp continuum, a pedestal of 1700 to 1900 e⁻
on HgAr and Ne and about 350 e⁻ on Xe, and it is not flat either: it falls toward the red on HgAr
and rises by an order of magnitude under the dense part of the Ne spectrum. The threshold finder
compares each pixel against `noise_factor` times its uncertainty with no continuum removed.
`find_arc_lines` therefore estimates the baseline with a sigma-clipped median and subtracts it
before detection, the same step `TiltCorrection.find_arc_lines` took in Chapter 6, so the
extractions can be passed as they are.

The uncertainty needs no attention either. `find_arc_lines` accepts any of the `astropy.nddata`
uncertainty types and converts to a standard deviation internally, so the `VarianceUncertainty` the
extraction carries is fine. Given a spectrum with no uncertainty at all, it estimates a single
per-pixel noise from the data with `specreduce.utils.utils.measure_noise`, a sigma-clipped form of
the DER_SNR estimator of
[Stoehr et al. (2008)](https://ui.adsabs.harvard.edu/abs/2008ASPC..394..505S/abstract). That is a
convenience for a quick look at a spectrum from somewhere else. On a line-rich arc the estimate
runs well above the propagated uncertainty, by a factor of two to five here, so a pipeline should
pass the uncertainty it propagated, as the book's spectra always do.

The 15-row aperture is wider than the raw frames would allow. A tilted line summed over many rows
smears across the columns it crosses and blunts the centroid, so on the raw detector the only safe
choice is a narrow band at the reference row. On the rectified frames the lines are vertical, rows
can be summed freely, and the width is set by signal-to-noise alone. Rectification is only
needed for this because the tilt distortion in the OSIRIS frames is strong; on an instrument with
little tilt a wide aperture is safe from the start.

## Finding the lines

`find_arc_lines` runs in three stages. It first estimates and removes the baseline, then calls
`find_lines_threshold` to locate peaks rising above `noise_factor` times the local uncertainty, and
finally fits a Gaussian to each detection within a window of `window * fwhm` pixels to refine the
center. It returns a `QTable`. It is `specreduce`'s one line detector, and the one
`WavelengthCalibration1D.find_lines` calls on your behalf; it is run directly here to see what it
does. What the calibration consumes in the end is the pixel positions, however they were
measured.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    ne_lines = find_arc_lines(spectra[1], fwhm=2.5, noise_factor=15)

print(f"columns: {ne_lines.colnames}")
print(ne_lines[:5])

Three columns: the refined `centroid` in pixels, the fitted `fwhm`, and the `amplitude`. The
centroid is what wavelength calibration consumes; the other two turn out to be the key to quality
control.

The two arguments look symmetrical but are not. `noise_factor` is the detection threshold, and it
is measured in units of the uncertainty rather than in electrons, so it has to be re-tuned after
any step that changes the noise model. Rectification is one: it propagates a sub-Poisson variance
(Chapter 6), the rectified uncertainty is about 0.79 of the raw one, and the same `noise_factor`
is therefore a 21 percent lower threshold after rectification than before. Lowering the threshold
is not automatically safe either. When more pixels clear it, adjacent lines can merge into a
single region, the Gaussian refit of the merged blob runs away, and the width filter below then
discards it, so a genuine bright line can vanish from the list because the threshold was
*lowered*; at `noise_factor=10` HgAr loses one of its brightest lines this way. Raising it too far
drops the faintest lines instead. The chapter uses **15** as the compromise between the two.

`fwhm` changes nothing about how many lines are found. It only sets the width of the window for
the Gaussian refit, so set it to the instrumental line width, about 2.5 pixels here as measured in
[Chapter 5](05-arc-lamps.ipynb): enough of each line to fit it, and not so much that the window
swallows its neighbors.

Not everything the threshold flags is a line. Noise spikes, the shoulders of bright lines and
blended pairs all get through, and the Gaussian refit exposes them, because a genuine arc line
comes back with a width close to the instrumental one. Keeping only detections whose fitted width
lies between 1.5 and 4 pixels removes the rest. Below that range are noise spikes and the flanks
of brighter lines, narrower than the instrument can deliver; above it are refits that failed and
spread across a broad feature, the worst here about 250 pixels. Neither has a meaningful centroid,
and the cut costs nothing: it drops 6 of 27 detections on HgAr, 8 of 31 on Ne and 3 of 31 on Xe.
The failed refits are also what astropy's convergence warnings complain about, so the warnings are
silenced rather than left to bury the output.

What survives, drawn on the three spectra, is what the rest of the calibration works with:

In [ ]:
detected = {}
for lamp, spectrum in zip(lamps, spectra):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        table = find_arc_lines(spectrum, fwhm=2.5, noise_factor=15)
    widths = np.asarray(table["fwhm"])
    detected[lamp] = table[(widths > 1.5) & (widths < 4.0)]

total = sum(len(detected[lamp]) for lamp in lamps)
print(f"detected across three lamps: {total} lines")

fig, axs = plt.subplots(3, figsize=(8, 5.0), sharex="all", constrained_layout=True)

for ax, lamp, spectrum in zip(axs, lamps, spectra):
    ax.plot(spectrum.flux.value, lw=0.6)
    for centroid in np.asarray(detected[lamp]["centroid"]):
        ax.axvline(centroid, c="C1", lw=0.5, alpha=0.7)
    ax.text(
        0.01,
        0.9,
        f"{lamp}  ({len(detected[lamp])} lines)",
        transform=ax.transAxes,
        va="top",
        size=8,
    )
    ax.set(yscale="log", ylim=(30, 1e7), ylabel="Flux [e$^-$]")

axs[-1].set(xlim=(0, nx - 1), xlabel="Dispersion [pix]");

HgAr and Xe come out as expected: a handful of bright, well-separated lines with a marker on nearly
every one, plus a run of faint lines at the red end of Xe that the threshold still reaches. Ne looks
different, and the difference is instructive. Its dense comb of bright lines between pixels 160
and 510 carries a single marker, at the brightest line near pixel 418, because
`find_lines_threshold` treats each contiguous run of pixels above the threshold as one line, and
in that comb every pixel clears it. The comb becomes one region, and the Gaussian refit settles on
its brightest member. The same happens to Xe's bright group between pixels 610 and 830, which
yields only the line near 749, and on a smaller scale to the close pairs in HgAr. The detections
are therefore the **isolated** lines, 72 of them across the three lamps, and that is what a
wavelength fit wants: an unresolved group would be hard to identify unambiguously in any case.

That is the detection half finished, each line with a sub-pixel centroid. Note that the horizontal
axis is still **pixels**. Nothing so far has used any physics.

## Reference line lists

The wavelengths come from a catalog. `specreduce` ships accessors for the well-curated
[`pypeit`](https://pypeit.readthedocs.io) line lists.

In [ ]:
catalogs = get_available_line_catalogs()
available = sorted(catalogs["pypeit"])
print(f"{len(available)} pypeit line lists available, including:")
print(
    "  "
    + ", ".join(
        n
        for n in available
        if n in ("ArI", "HgI", "NeI", "XeI", "CdI", "HeI", "ThAr", "OH_GMOS")
    )
)

The names are *ions*, not lamps, so an HgAr lamp needs `HgI` and `ArI` together. Note also the
`OH_*` entries: in the near-infrared the sky's own OH airglow is used as the wavelength reference,
which is why the docstring speaks of "lamps" even when the source is the atmosphere.

:::{note}
`load_pypeit_calibration_lines` **downloads** the lists on first use and caches them locally, so
this cell needs network access the first time it runs.
:::

In [ ]:
lamp_ions = {"HgAr": ["HgI", "ArI"], "Ne": ["NeI"], "Xe": ["XeI"]}
grism_range = (5000, 10500) * u.AA

catalog = {}
for lamp, ions in lamp_ions.items():
    lines = load_pypeit_calibration_lines(ions)["wavelength"].to(u.AA)
    inside = (lines > grism_range[0]) & (lines < grism_range[1])
    catalog[lamp] = lines[inside]
    print(
        f"{lamp:>5}: {len(lines):4d} catalog lines, {len(catalog[lamp]):4d} within "
        f"{grism_range[0].value:.0f}-{grism_range[1].value:.0f} A, "
        f"{len(detected[lamp]):3d} detected"
    )

The mismatch in those numbers runs in both directions. HgAr offers far more catalog lines in
range than were detected: most catalog entries are too faint to appear in a short exposure, so
the detections are a sparse, unlabeled subset of a much denser list. Xe is the opposite case,
with more detections than catalog entries, because the `XeI` list is thin at these wavelengths,
so some genuine detections have **no** catalog counterpart at all.

Neither situation is unusual, and together they mean that nothing about the two lists lines up by
itself. Which detection goes with which catalog line has to be worked out, knowing that many on
both sides have no partner.

### Instrument-specific line lists

A generic catalog lists every transition the species can emit. An observatory that supports a
spectrograph usually publishes something more useful: a line list per lamp and grism containing
the lines that instrument actually shows at its resolution, typically derived from its own
calibration exposures. GTC does this for OSIRIS, and the list for HgAr through R1000R is short
enough to keep in `common.py`:

In [ ]:
print(f"GTC HgAr line list for R1000R: {len(GTC_HGAR)} lines (air wavelengths)")
print(GTC_HGAR)
print(f"\npypeit HgI + ArI lines in the same range: {len(catalog['HgAr'])}")

Eleven lines against 142. The generic list is not wrong, but at this resolution and exposure most
of its entries never appear, and a list of the lines that do is worth far more than a list of the
lines that could. The GTC wavelengths are in air and the `pypeit` lists are in vacuum, which
matters when the two are combined in Chapter 10.

## Identifying lines by hand

Detection produced positions in pixels; the catalog holds wavelengths. Pairing the two is the
identification, and in the interactive workflow it is done by a person, with a **reference
figure**: a plot of the same lamp's spectrum through the same instrument and grism, on a
wavelength axis, with the lines labeled. Observatories publish them for the configurations they
support, lamp manufacturers publish spectra of their lamps, and arc atlases from other instruments
at similar resolution serve when nothing closer exists. For OSIRIS, GTC's
[arc line maps](https://www.gtc.iac.es/instruments/osiris/osiris.php) include one figure per lamp
and grism, and the R1000R maps for
[HgAr](https://www.gtc.iac.es/instruments/osiris/media/R1000R-HgAr.jpg),
[Ne](https://www.gtc.iac.es/instruments/osiris/media/R1000R-Ne.jpg) and
[Xe](https://www.gtc.iac.es/instruments/osiris/media/R1000R-Xe.jpg) are the references for this
dataset.

The HgAr map is easy to read. One line dominates everything, the Hg green line at 5460.735 Å,
alone at the blue end. A close pair of Hg lines follows at 5769.598 and 5790.663 Å. Then comes a
cluster of Ar lines between 6965 and 8521 Å with two bright pairs in it, 7635.106 with 7724.207 Å
and 8115.311 with 8264.522 Å, and the red end holds a few isolated Ar lines of which 9122.967 Å is
the strongest.

The observed side of the comparison is the detected-line list, drawn on the spectrum with each
line labeled by its pixel position. The lower panel stands in for the reference figure: it is the
GTC list drawn on a wavelength axis, which is what a catalog gives you when no figure exists.

In [ ]:
hgar = detected["HgAr"]
centroids = np.asarray(hgar["centroid"])
flux = spectra[0].flux.value

fig, axs = plt.subplots(
    2, figsize=(8, 5.4), height_ratios=[3, 1], constrained_layout=True
)

ax = axs[0]
ax.plot(flux, lw=0.6, c="0.2")
for centroid in centroids:
    ax.axvline(centroid, c="C1", lw=0.5, alpha=0.6)
    ax.text(
        centroid,
        1.8 * flux[int(round(centroid))],
        f"{centroid:.0f}",
        rotation=90,
        ha="center",
        va="bottom",
        size=6,
    )
ax.set(
    yscale="log",
    xlim=(0, nx - 1),
    ylim=(300, 3e7),
    xlabel="Dispersion [pix]",
    ylabel="Flux [e$^-$]",
    title="HgAr, detected lines labeled by pixel",
)

ax = axs[1]
ax.vlines(GTC_HGAR, 0, 1, color="C0", lw=1.0)
for i, wavelength in enumerate(GTC_HGAR):
    # Nudge the labels of lines closer than 50 A apart so they do not overlap.
    close_below = i > 0 and wavelength - GTC_HGAR[i - 1] < 50
    close_above = i < len(GTC_HGAR) - 1 and GTC_HGAR[i + 1] - wavelength < 50
    nudge = 45 if close_below else (-45 if close_above else 0)
    ax.text(
        wavelength + nudge,
        1.08,
        f"{wavelength:.3f}",
        rotation=90,
        ha="center",
        va="bottom",
        size=6,
    )
ax.set(
    xlim=(5000, 10500),
    ylim=(0, 3.2),
    yticks=[],
    xlabel=r"Wavelength [$\AA$]",
    title="GTC HgAr line list for R1000R",
)
for ax in axs:
    ax.title.set_size(8)

The real reference carries the brightnesses as well, and they are what make the identification
quick. Read the observed panel against the map and the pairing follows:

- The brightest detection by a wide margin, near pixel 87 with nothing else within three hundred
  pixels, can only be the green line at 5460.735 Å.
- The brightest lines in the middle of the frame, near pixels 533 and 550, form a close pair with
  the first the stronger, which is the Ar pair 7635.106 and 7724.207 Å on the map. Their
  separation, 17 pixels for 89 Å, gives a first estimate of the dispersion, and every further
  pairing has to be consistent with it.
- The next bright pair, near pixels 622 and 650, is 8115.311 and 8264.522 Å, and the bright,
  isolated line near pixel 803 is 9122.967 Å. Both fall where the dispersion from the first pair
  says they should.

The selection rule is to take lines that are **strong and isolated in both** the spectrum and the
reference, and to match the **pattern** of spacings and relative brightnesses rather than any
single line. It is equally a matter of what to leave out. The Hg pair at 5769.598 and 5790.663 Å is
bright on the map but only four pixels apart at this dispersion, blended at this resolution, and
no detection survived the width filter there. The fainter Ar lines between the bright pairs are
real, but a faint line is easier to confuse with a neighbor, and the refinement in Chapter 10
recovers them once a solution exists.

Those six pairs are the result of the exercise. They are stored in `common.py` as `ANCHOR_PIXELS`
and `ANCHOR_WAVELENGTHS`, with the pixels rounded to whole numbers because that is the precision a
plot can be read to; Chapter 10 snaps them to the measured centroids. Here they are, drawn at the
detection each one snaps to:

In [ ]:
nearest = centroids[np.abs(centroids[:, None] - ANCHOR_PIXELS[None, :]).argmin(axis=0)]

fig, ax = plt.subplots(figsize=(8, 2.6), constrained_layout=True)
ax.plot(flux, lw=0.6, c="0.2")
for centroid, wavelength in zip(nearest, ANCHOR_WAVELENGTHS):
    ax.axvline(centroid, c="C3", lw=0.8)
    ax.text(
        centroid,
        1.8 * flux[int(round(centroid))],
        f"{wavelength:.3f}",
        rotation=90,
        ha="center",
        va="bottom",
        size=6,
    )
ax.set(
    yscale="log",
    xlim=(0, nx - 1),
    ylim=(300, 3e7),
    xlabel="Dispersion [pix]",
    ylabel="Flux [e$^-$]",
);

A quick check closes the loop. This is not a step of the reduction, and in practice one would go
straight to the fit in Chapter 10; it is here to show, before anything is built on them, that the
six pairs agree with each other. A quadratic through them is a crude solution, but it is enough to
predict where every other line on the GTC list should fall:

In [ ]:
quadratic = np.polyfit(nearest, ANCHOR_WAVELENGTHS, 2)
grid = np.arange(nx)
predicted = np.interp(GTC_HGAR, np.polyval(quadratic, grid), grid)

print(
    f"{'wavelength [A]':>14}  {'predicted pixel':>15}  {'nearest detection':>17}  "
    f"{'offset [pix]':>12}"
)
for wavelength, pixel in zip(GTC_HGAR, predicted):
    offset = centroids - pixel
    j = np.abs(offset).argmin()
    tag = "anchor" if wavelength in ANCHOR_WAVELENGTHS else ""
    print(
        f"{wavelength:>14.3f}  {pixel:>15.1f}  {centroids[j]:>17.2f}  {offset[j]:>12.1f}  {tag}"
    )

Every line the width filter kept lands within two pixels of where the quadratic puts it, the three
Ar lines that were not anchors included, and the blended Hg pair is predicted at pixels 155 and
160, where no detection survived. The identification is self-consistent. That is as far as the
work by hand needs to go: the fit itself, with all the lines, is Chapter 10's.

## What this means for the next chapters

Detection is a solved problem. `find_arc_lines` returns a sub-pixel centroid for every isolated
line on every lamp, and its two other columns, the fitted width and the amplitude, are what tell
a real line from a spurious one. This chapter ran the detector by hand and applied a width cut to
look at what it returns; from here on `WavelengthCalibration1D` runs it itself, through
`find_lines`, and the tolerance of the line matching keeps the spurious detections out of the
fit. Positions from another detection routine go into the same class in place of its own.

Identification, done by hand against a reference figure, gives six pairs that are certainly
right. Six pairs do not make a wavelength solution, but they make one reachable. Fit a low-order
polynomial through them, use it to predict where every catalog line falls, match the detections
that land close enough, refit with all of them, tighten the tolerance, repeat. That loop is what
`WavelengthCalibration1D` implements, and
[Chapter 10](10-wavelength-solution-interactive.ipynb) runs it step by step, inspecting the
residuals as it goes.

[Chapter 11](11-wavelength-solution-automated.ipynb) takes a different route in, for a different
situation. A pipeline reducing many nights from a stable instrument already knows the solution to
within a small range, because the solution depends on the instrument setup rather than on the
night. With that prior in hand, `fit_dispersion` searches for the solution inside it, and no
anchors are needed.

## Summary

- **Detection and identification are different problems.** `find_arc_lines` solves the first
  reliably, normally called for you by `WavelengthCalibration1D.find_lines`, and pixel positions
  from another detection routine serve the calibration just as well. The second is where reductions go
  wrong.
- **Rectify first if the tilt calls for it.** A tilted line smears when summed over many rows, so
  on a strongly tilted frame the extraction aperture has to stay narrow; once the frame is
  rectified the lines are vertical and the aperture width is set by signal-to-noise alone. On an
  instrument with little tilt a wide aperture is safe from the start.
- **`noise_factor` is the detection threshold, in units of the uncertainty**, so re-tune it
  whenever the noise model changes. Lowering it can lose bright lines by merging neighbors,
  raising it drops the faint ones. `fwhm` only sizes the refit window; set it to the instrumental
  line width.
- **The fitted width tells a line from a spurious detection.** Genuine lines come back near the
  instrumental width; noise spikes are narrower and failed refits far wider. Unresolved groups
  collapse to a single detection, so what the detector returns are the isolated lines.
- **Identification is a comparison against a reference figure**, the detected lines labeled in
  pixels against the same lamp through the same setup labeled in wavelength. Choose strong,
  isolated lines present in both, match the pattern of spacings and brightnesses rather than
  single lines, and leave blends and faint lines to the refinement.
- **Use an instrument-specific line list when one exists.** GTC's eleven-line HgAr list holds the
  lines this setup shows where the generic catalogs hold 142, and nothing in a generic catalog
  lines up with the detections by itself. Mind air against vacuum wavelengths when lists are
  combined.

The six anchors are stored in `common.py` as `ANCHOR_PIXELS` and `ANCHOR_WAVELENGTHS`, and
Chapter 10 imports them.

Next: [Interactive Wavelength Solution](10-wavelength-solution-interactive.ipynb).